# Round trip of a sales representative through 14 cities

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ula-uab/lscm-decision-making/blob/main/cases/traveling_salesman/notebooks/traveling_salesman.ipynb)

A sales representative leaves Hamburg, visits 13 other cities of northern Germany once each and returns to Hamburg. She looks for the shortest route. The notebook follows the search in steps:

1. The cities, the distances and the number of possible routes.
2. A route built with the nearest neighbor: each time, the nearest city not yet visited.
3. A route built with the sweep: the cities in the order of their angle seen from Hamburg.
4. A local search that shortens a route one move at a time, with two moves: the swap of two adjacent cities and the reversal of a sub-path.
5. The shortest route, calculated exactly, and where the local search stops from many random starting routes.

## How to use the notebook

Run the cells in order, with ▶ or *Shift + Enter*. Some cells have a form (drop-down lists, tick boxes and sliders): change the values and run the cell again. Outside Colab the form is a line such as `seed = 1`: edit the value and run the cell.

All the data and results are explained in the [document of the case](https://github.com/ula-uab/lscm-decision-making/blob/main/cases/traveling_salesman/traveling_salesman.md). The data are from Ivanov et al. (2025, §14.3). Running the notebook is optional: it is support material, not part of the assessment.

In [ ]:
# @title Setup (run this cell first) { display-mode: "form" }
# In Google Colab, install the case from GitHub (it takes a few seconds).
# On your own computer, install the repository environment instead (see README).
import sys
if "google.colab" in sys.modules:
    url = "git+https://github.com/ula-uab/lscm-decision-making#subdirectory=cases/traveling_salesman"
    %pip install -q {url}
    # A session that already had the case keeps its old code unless it is reinstalled
    %pip install -q --force-reinstall --no-deps {url}
    # and unless its old code is removed from memory, so that the import below reads
    # the new one: no need to restart the session
    import importlib
    for name in [m for m in sys.modules if m == "routes" or m.startswith("routes.")]:
        del sys.modules[name]
    importlib.invalidate_caches()
import routes as rt

## 1. Cities, distances and number of routes

The first table gives the 14 cities, their abbreviations and their coordinates. The second gives the distance between each pair of cities, in kilometers; a route of the sales representative is a sequence of cities that starts and ends in Hamburg (HH) and visits each of the other 13 once, and its length is the sum of the distances between consecutive cities. The cell also counts the possible routes and draws the cities on a map, north up and with the same scale on both axes.

How long would it take to calculate the length of every route at a million routes a second?

In [ ]:
# @title Data { display-mode: "form" }
rt.show_data()

## 2. Nearest neighbor

A heuristic gives a feasible route by following a reasonable strategy, without proving that no route is shorter. A construction heuristic builds a first route, one city at a time. The nearest neighbor starts in Hamburg and goes each time to the nearest city not yet visited; when every city has been visited, it returns to Hamburg. It is greedy: each step takes the option that looks best at that moment, the shortest leg available.

The cell gives the route leg by leg, with the distance of each leg and the distance so far, and draws it on the map.

Which legs of the map would you change by hand, and why did the nearest neighbor choose them?

In [ ]:
# @title Nearest neighbor { display-mode: "form" }
rt.nearest_neighbor()

## 3. Sweep

The sweep orders the cities by the angle of the line that joins them with Hamburg, measured from north counterclockwise, and visits them in that order. It comes from the routes of several vehicles with a limited capacity, where each vehicle takes the next cities of the order until it is full; here a single vehicle takes them all. The sweep is a construction heuristic, and it is not greedy: the order is fixed by the angles before the first leg, and no distance is compared.

The cell gives the angle of each city in the order of the sweep, the route and its map; the grey lines join Hamburg with each city, and the dotted arrow marks north, the angle 0.

Would the sweep give a short route if the cities were spread along a line that passes through Hamburg?

In [ ]:
# @title Sweep { display-mode: "form" }
rt.sweep()

## 4. Local search

An improvement heuristic starts from a feasible route and changes it to make it shorter. A local search changes it one move at a time. A move changes a route into another route; the neighborhood of a route is the set of routes that one move reaches. Both moves of the case reverse the cities of the route between two positions:

- **swap of two adjacent cities:** two cities next to each other change places; 12 moves;
- **reversal of a sub-path:** the cities from one position to another are visited in the opposite order; 78 moves.

Reversing a sub-path takes out the two arcs at its ends and puts in two new ones; the arcs inside it stay, traveled the other way, with the same length.

### Pass of Ivanov et al.

Ivanov et al. (2025, §14.3.3) shorten the route of the nearest neighbor with one pass of swaps. For each position from 1 to 12, they compare the four cities around it with the same four after swapping the city at that position with the next one, and keep the swap if it is shorter. The cell gives the twelve comparisons, in km, and the route after the pass.

The pass looks at each position once. Would a second pass over the new route find another swap that shortens it?

In [ ]:
# @title Pass of Ivanov et al. { display-mode: "form" }
rt.ivanov_pass()

### Local search with best improvement

At each step, the search evaluates every move of the neighborhood and makes the one that leaves the shortest route, if it is shorter than the current one. It stops when no move shortens the route: the route where it stops is a local optimum, a route with no shorter route in its neighborhood.

The form sets the starting route (the nearest neighbor, the sweep, or a random route, whose seed sets which one) and the move. The table gives each step: the sub-path reversed, in its order before the move, the arcs that go out and come in, and the length of the route. The cell then gives the best move from the route where the search stops, and how far that route is from the shortest route of section 5. With *step_by_step* ticked, a map of each step shows in dashed red the arcs that go out and in green those that come in; the last map is the route where the search stops.

From the nearest neighbor, does the search stop at the same route with each move? Try several seeds: does a random starting route stop at the same length as the nearest neighbor?

In [ ]:
# @title Local search { display-mode: "form" }
start = "nearest neighbor"  # @param ["nearest neighbor", "sweep", "random"]
seed = 1  # @param {type:"integer"}
move = "reversal of a sub-path"  # @param ["swap of two adjacent cities", "reversal of a sub-path"]
step_by_step = False  # @param {type:"boolean"}
rt.local_search_steps(start, move, seed, step_by_step)

## 5. Shortest route and local optima

A heuristic does not prove that its route is the shortest. With 13 cities, the shortest route can be calculated exactly, without a solver, by dynamic programming over the sets of cities: for each set of cities and each city of the set, the shortest path that leaves Hamburg, visits the set and ends at that city, built from the sets with one city fewer. The cell gives the length of the shortest route, every route with that length, the length of each route of the previous sections and how far it is above the shortest, and the maps of the shortest routes.

Which of the routes of the previous sections are as short as the shortest route, and which heuristic came closest without a local search?

In [ ]:
# @title Shortest route { display-mode: "form" }
rt.shortest_route()

### Local optima from random starting routes

The search with the reversal of a sub-path is started from many routes that visit the cities in random order. The table gives, for each length where the search stops, how many starting routes stop there and their share; in the bar chart, the green bar is the shortest route. The slider sets the number of random starting routes; 20,000 take a few seconds.

What share of the starts stop above the shortest route? If a company could run the search from only one starting route, what would it know about the route it gets?

In [ ]:
# @title Local optima { display-mode: "form" }
starts = 20000  # @param {type:"slider", min:1000, max:50000, step:1000}
rt.local_optima(starts)

All the numbers are explained in the [document of the case](https://github.com/ula-uab/lscm-decision-making/blob/main/cases/traveling_salesman/traveling_salesman.md), which also has two scripts that reproduce them.